
# Grammar Correction Prompt Experiment from Scratch

このノートブックは、英語文法修正タスクに対して

- 条件A: **「文法的な間違いがあります」**
- 条件B: **「文法的な間違いがあれば」**

の差を、**出力評価**と**レイヤー解析**の両方から比較するための、最初から実行可能な実験ノートブックです。

## この版で入っているもの

### 出力評価
- exact match
- unchanged from input / changed
- embedding cosine（意味保持）
- optional grammar classifier

### レイヤー解析
- prompt / output の層別スコア
- 正文ケース / 誤文ケースを分けた可視化
- A-B 差分曲線
- 最大差 layer の要約表

## 方針
- 既存CSVは前提にしない
- データセット定義から保存まで、この notebook 単体で完結
- instruct モデルでも過剰修正を抑えるため、A/B **両方に同じ出力制約**を加える


In [ ]:

# Cell 1: インストール（必要に応じて実行）
# !pip -q install -U transformers sentence-transformers scikit-learn scipy matplotlib pandas numpy accelerate


In [ ]:

# Cell 2: 基本ライブラリ
import os
import re
import json
import math
import random
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
)

from sentence_transformers import SentenceTransformer
from scipy.stats import ttest_rel

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


In [ ]:
# Cell 3: 実験設定
from pathlib import Path
import shutil
import zipfile

MODEL_LIST = [
    {
        "name": "Qwen/Qwen2.5-0.5B",
        "label": "qwen_0.5b_base",
        "instruction_tuned": False,
    },
    {
        "name": "Qwen/Qwen2.5-1.5B-Instruct",
        "label": "qwen_1.5b_instruct",
        "instruction_tuned": True,
    },
]

MAX_NEW_TOKENS = 64
TEMPERATURE = 0.0
DO_SAMPLE = False

RUN_NAME = "grammar_prompt_experiment_run"
BASE_OUTPUT_DIR = Path("./grammar_prompt_experiment_bundle")
RUN_DIR = BASE_OUTPUT_DIR / RUN_NAME
DATA_DIR = RUN_DIR / "data"
FIG_DIR = RUN_DIR / "figures"
TABLE_DIR = RUN_DIR / "tables"
LOG_DIR = RUN_DIR / "logs"

for p in [BASE_OUTPUT_DIR, RUN_DIR, DATA_DIR, FIG_DIR, TABLE_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

USE_GRAMMAR_CLASSIFIER = False
GRAMMAR_MODEL_NAME = "textattack/roberta-base-CoLA"

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

SYSTEM_PROMPT = "You are a careful English grammar correction assistant."
COMMON_OUTPUT_CONSTRAINT = (
    "If the sentence is already grammatically correct, return the original sentence unchanged. "
    "Do not add any explanation. Output only the final English sentence."
)

PROMPT_CONDITIONS = [
    {
        "name": "A_assert_error",
        "instruction": (
            "Look at the following English sentence. It contains a grammatical error. "
            "Rewrite it as a correct English sentence. "
            + COMMON_OUTPUT_CONSTRAINT
        ),
    },
    {
        "name": "B_if_error",
        "instruction": (
            "Look at the following English sentence. If it contains a grammatical error, "
            "rewrite it as a correct English sentence. "
            + COMMON_OUTPUT_CONSTRAINT
        ),
    },
]

print("BASE_OUTPUT_DIR:", BASE_OUTPUT_DIR.resolve())
print("RUN_DIR:", RUN_DIR.resolve())

In [ ]:

# Cell 4: データセット定義（必要なら増やしてください）
# sentence: 入力文
# gold: 期待される最終文（誤りがあれば修正文、正文なら原文）
# has_error: 元文に文法誤りがあるか

DATA = [
    # Un grammatical / error cases
    {"id": "e01", "sentence": "She go to school every day.", "gold": "She goes to school every day.", "has_error": True},
    {"id": "e02", "sentence": "I have ate breakfast already.", "gold": "I have eaten breakfast already.", "has_error": True},
    {"id": "e03", "sentence": "He don't like carrots.", "gold": "He doesn't like carrots.", "has_error": True},
    {"id": "e04", "sentence": "There is many books on the table.", "gold": "There are many books on the table.", "has_error": True},
    {"id": "e05", "sentence": "If I will see him, I will tell him.", "gold": "If I see him, I will tell him.", "has_error": True},
    {"id": "e06", "sentence": "The information are very useful.", "gold": "The information is very useful.", "has_error": True},
    {"id": "e07", "sentence": "My brother can sings very well.", "gold": "My brother can sing very well.", "has_error": True},
    {"id": "e08", "sentence": "She enjoys to read novels.", "gold": "She enjoys reading novels.", "has_error": True},
    {"id": "e09", "sentence": "Yesterday I go to the library.", "gold": "Yesterday I went to the library.", "has_error": True},
    {"id": "e10", "sentence": "This book is more easier than that one.", "gold": "This book is easier than that one.", "has_error": True},
    {"id": "e11", "sentence": "He suggested me to apply early.", "gold": "He suggested that I apply early.", "has_error": True},
    {"id": "e12", "sentence": "The woman which lives next door is a doctor.", "gold": "The woman who lives next door is a doctor.", "has_error": True},

    # Grammatical / correct cases
    {"id": "c01", "sentence": "She goes to school every day.", "gold": "She goes to school every day.", "has_error": False},
    {"id": "c02", "sentence": "I have already eaten breakfast.", "gold": "I have already eaten breakfast.", "has_error": False},
    {"id": "c03", "sentence": "He doesn't like carrots.", "gold": "He doesn't like carrots.", "has_error": False},
    {"id": "c04", "sentence": "There are many books on the table.", "gold": "There are many books on the table.", "has_error": False},
    {"id": "c05", "sentence": "If I see him, I will tell him.", "gold": "If I see him, I will tell him.", "has_error": False},
    {"id": "c06", "sentence": "The information is very useful.", "gold": "The information is very useful.", "has_error": False},
    {"id": "c07", "sentence": "My brother can sing very well.", "gold": "My brother can sing very well.", "has_error": False},
    {"id": "c08", "sentence": "She enjoys reading novels.", "gold": "She enjoys reading novels.", "has_error": False},
    {"id": "c09", "sentence": "Yesterday I went to the library.", "gold": "Yesterday I went to the library.", "has_error": False},
    {"id": "c10", "sentence": "This book is easier than that one.", "gold": "This book is easier than that one.", "has_error": False},
    {"id": "c11", "sentence": "He suggested that I apply early.", "gold": "He suggested that I apply early.", "has_error": False},
    {"id": "c12", "sentence": "The woman who lives next door is a doctor.", "gold": "The woman who lives next door is a doctor.", "has_error": False},
]

df_data = pd.DataFrame(DATA)
display(df_data.head())
print("n_total =", len(df_data), "| n_error =", int(df_data["has_error"].sum()), "| n_correct =", int((~df_data["has_error"]).sum()))


In [ ]:

# Cell 5: ユーティリティ
def safe_text(text, fallback=""):
    if text is None:
        return fallback
    text = str(text).strip()
    return text if text else fallback

def normalize_sentence(text):
    text = safe_text(text)
    text = text.replace("’", "'").replace("“", '"').replace("”", '"')
    text = re.sub(r"^Corrected sentence\s*[:：]\s*", "", text, flags=re.I)
    text = re.sub(r"^The corrected sentence is\s*[:：]?\s*", "", text, flags=re.I)
    text = re.sub(r"^Output\s*[:：]\s*", "", text, flags=re.I)
    text = text.split("\n")[0].strip()
    text = re.sub(r"\s+", " ", text).strip()
    return text

def build_user_prompt(instruction, sentence):
    return (
        f"{instruction}\n\n"
        f"{COMMON_OUTPUT_RULE}\n\n"
        f"Sentence: {sentence}\n\n"
        f"Final sentence:"
    )

def normalize_vec(v, eps=1e-8):
    v = np.asarray(v, dtype=np.float32)
    n = np.linalg.norm(v)
    return v if n < eps else v / n

def cosine_sim(a, b):
    a = normalize_vec(a)
    b = normalize_vec(b)
    return float(np.dot(a, b))

def lexical_change_ratio(src, pred):
    src_tokens = re.findall(r"[A-Za-z']+", safe_text(src).lower())
    pred_tokens = re.findall(r"[A-Za-z']+", safe_text(pred).lower())
    if len(src_tokens) == 0 and len(pred_tokens) == 0:
        return 0.0
    overlap = len(set(src_tokens) & set(pred_tokens))
    denom = max(len(set(src_tokens) | set(pred_tokens)), 1)
    return 1.0 - (overlap / denom)

def paired_effect_size(x, y):
    diff = np.asarray(x, dtype=float) - np.asarray(y, dtype=float)
    if len(diff) < 2:
        return np.nan
    sd = diff.std(ddof=1)
    if sd == 0:
        return np.nan
    return diff.mean() / sd

def paired_test_from_pivot(df, col_a, col_b, label):
    sub = df[[col_a, col_b]].dropna().copy()
    if len(sub) < 2:
        return {
            "comparison": label,
            "n": len(sub),
            "mean_A_minus_B": np.nan,
            "t": np.nan,
            "p": np.nan,
            "cohen_d_paired": np.nan,
        }
    x = sub[col_a].astype(float).values
    y = sub[col_b].astype(float).values
    res = ttest_rel(x, y)
    return {
        "comparison": label,
        "n": len(sub),
        "mean_A_minus_B": float(np.mean(x - y)),
        "t": float(res.statistic),
        "p": float(res.pvalue),
        "cohen_d_paired": float(paired_effect_size(x, y)),
    }


In [ ]:

# Cell 6: モデル読み込み
@dataclass
class ModelBundle:
    config: dict
    tokenizer: object
    model: object

def load_model_bundle(model_cfg):
    tokenizer = AutoTokenizer.from_pretrained(model_cfg["name"], trust_remote_code=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_cfg["name"],
        torch_dtype="auto",
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()
    return ModelBundle(config=model_cfg, tokenizer=tokenizer, model=model)

embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)

grammar_tokenizer = None
grammar_model = None
if USE_GRAMMAR_CLASSIFIER:
    grammar_tokenizer = AutoTokenizer.from_pretrained(GRAMMAR_MODEL_NAME)
    grammar_model = AutoModelForSequenceClassification.from_pretrained(GRAMMAR_MODEL_NAME).to(DEVICE)
    grammar_model.eval()

print("Embedding model ready:", EMBED_MODEL_NAME)
print("Grammar classifier enabled:", USE_GRAMMAR_CLASSIFIER)


In [ ]:

# Cell 7: 生成・hidden state 取得
def format_prompt(bundle, user_prompt, system_prompt=SYSTEM_PROMPT):
    tokenizer = bundle.tokenizer
    if hasattr(tokenizer, "apply_chat_template") and bundle.config["instruction_tuned"]:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return f"{system_prompt}\n\n{user_prompt}"

def tokenize_to_device(bundle, text):
    inputs = bundle.tokenizer(text, return_tensors="pt", truncation=True)
    return {k: v.to(bundle.model.device) for k, v in inputs.items()}

def get_hidden_states_last_token(bundle, text):
    text = safe_text(text, fallback=".")
    inputs = tokenize_to_device(bundle, text)
    with torch.no_grad():
        outputs = bundle.model(**inputs, output_hidden_states=True, use_cache=False)
    last_idx = inputs["input_ids"].shape[1] - 1
    out = []
    for hs in outputs.hidden_states:
        vec = hs[0, last_idx, :].detach().float().cpu().numpy()
        out.append(normalize_vec(vec))
    return out

def get_prompt_end_hidden_states(bundle, user_prompt):
    full_prompt = format_prompt(bundle, user_prompt)
    return get_hidden_states_last_token(bundle, full_prompt)

def generate_with_scores(bundle, user_prompt):
    full_prompt = format_prompt(bundle, user_prompt)
    inputs = tokenize_to_device(bundle, full_prompt)

    with torch.no_grad():
        generated = bundle.model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            temperature=TEMPERATURE,
            pad_token_id=bundle.tokenizer.pad_token_id,
            eos_token_id=bundle.tokenizer.eos_token_id,
            return_dict_in_generate=True,
            output_scores=True,
        )

    full_ids = generated.sequences[0]
    prompt_len = inputs["input_ids"].shape[1]
    gen_ids = full_ids[prompt_len:]
    raw_text = bundle.tokenizer.decode(gen_ids, skip_special_tokens=True)
    pred = normalize_sentence(raw_text)

    mean_logprob = np.nan
    if generated.scores and len(gen_ids) > 0:
        token_logps = []
        for step, scores in enumerate(generated.scores):
            if step >= len(gen_ids):
                break
            target_id = gen_ids[step]
            logp = F.log_softmax(scores[0], dim=-1)[target_id].item()
            token_logps.append(logp)
        if token_logps:
            mean_logprob = float(np.mean(token_logps))

    return {
        "prompt_text": full_prompt,
        "raw_generated": raw_text,
        "generated": pred,
        "mean_logprob": mean_logprob,
    }


In [ ]:

# Cell 8: アンカー定義（レイヤー解析）
ANCHOR_TEXTS = {
    "correction_mode": [
        "The sentence definitely contains a grammatical error and must be corrected.",
        "A correction is clearly required.",
        "This input needs revision.",
        "The sentence is not grammatically correct.",
        "Correction is necessary here.",
    ],
    "conditional_mode": [
        "If the sentence contains an error, correct it; otherwise leave it unchanged.",
        "Correction is only necessary when an actual error exists.",
        "The system should first decide whether correction is needed.",
        "If the sentence is already correct, do not revise it.",
        "Revision depends on whether an error is present.",
    ],
    "high_confidence": [
        "The answer is clear and can be given confidently.",
        "I am confident that the correct action is obvious.",
        "This case is straightforward.",
        "There is little uncertainty here.",
        "The decision is clear.",
    ],
    "low_confidence": [
        "It is uncertain whether correction is needed.",
        "This case may or may not require revision.",
        "The need for correction is not fully clear.",
        "I am not sure whether the sentence should be changed.",
        "This judgment is uncertain.",
    ],
}

def build_anchor_vectors(bundle, texts):
    layer_lists = [get_hidden_states_last_token(bundle, t) for t in texts]
    n_layers = len(layer_lists[0])
    out = []
    for layer_idx in range(n_layers):
        stack = np.stack([layer_lists[i][layer_idx] for i in range(len(layer_lists))], axis=0)
        out.append(normalize_vec(stack.mean(axis=0)))
    return out

def score_against_anchor(layer_vecs, anchor_vecs):
    return [cosine_sim(layer_vecs[i], anchor_vecs[i]) for i in range(len(layer_vecs))]


In [ ]:

# Cell 9: 評価関数
def semantic_similarity(a, b):
    emb = embed_model.encode([safe_text(a), safe_text(b)], convert_to_numpy=True, normalize_embeddings=True)
    return float(np.dot(emb[0], emb[1]))

def grammar_acceptability_score(text):
    if not USE_GRAMMAR_CLASSIFIER:
        return np.nan
    inputs = grammar_tokenizer(
        safe_text(text, fallback="."),
        return_tensors="pt",
        truncation=True,
        padding=True,
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        logits = grammar_model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0].detach().cpu().numpy()
    # CoLA binary classifier を想定: acceptability-positive の確率を返す
    if probs.shape[0] == 2:
        return float(probs[1])
    return float(np.max(probs))

def evaluate_output(src, gold, pred, has_error):
    src_n = normalize_sentence(src)
    gold_n = normalize_sentence(gold)
    pred_n = normalize_sentence(pred)

    exact_match_gold = int(pred_n == gold_n)
    unchanged_from_input = int(pred_n == src_n)
    changed = int(pred_n != src_n)

    corrected_when_needed = int(has_error and pred_n == gold_n)
    preserved_when_correct = int((not has_error) and pred_n == src_n)

    semantic_to_gold = semantic_similarity(pred_n, gold_n)
    semantic_to_input = semantic_similarity(pred_n, src_n)

    return {
        "pred_norm": pred_n,
        "exact_match_gold": exact_match_gold,
        "unchanged_from_input": unchanged_from_input,
        "changed": changed,
        "corrected_when_needed": corrected_when_needed,
        "preserved_when_correct": preserved_when_correct,
        "semantic_to_gold": semantic_to_gold,
        "semantic_to_input": semantic_to_input,
        "lexical_change_ratio": lexical_change_ratio(src_n, pred_n),
        "grammar_acceptability": grammar_acceptability_score(pred_n),
    }


In [ ]:

# Cell 10: 実験実行
all_outputs = []
all_layers = []

for model_cfg in MODEL_LIST:
    print("=" * 100)
    print("Loading model:", model_cfg["name"])
    bundle = load_model_bundle(model_cfg)

    anchor_vectors = {
        name: build_anchor_vectors(bundle, texts)
        for name, texts in ANCHOR_TEXTS.items()
    }

    for ex in DATA:
        for cond in PROMPT_CONDITIONS:
            user_prompt = build_user_prompt(cond["instruction"], ex["sentence"])
            gen_pack = generate_with_scores(bundle, user_prompt)
            pred = gen_pack["generated"]

            eval_pack = evaluate_output(
                src=ex["sentence"],
                gold=ex["gold"],
                pred=pred,
                has_error=ex["has_error"],
            )

            prompt_vecs = get_prompt_end_hidden_states(bundle, user_prompt)
            output_basis = pred if len(pred) > 0 else ex["sentence"]
            output_vecs = get_hidden_states_last_token(bundle, output_basis)

            prompt_scores = {k: score_against_anchor(prompt_vecs, v) for k, v in anchor_vectors.items()}
            output_scores = {k: score_against_anchor(output_vecs, v) for k, v in anchor_vectors.items()}

            row = {
                "model_name": model_cfg["name"],
                "model_label": model_cfg["label"],
                "instruction_tuned": model_cfg["instruction_tuned"],
                "example_id": ex["id"],
                "sentence": ex["sentence"],
                "gold": ex["gold"],
                "has_error": ex["has_error"],
                "condition": cond["condition"],
                "instruction": cond["instruction"],
                "user_prompt": user_prompt,
                "raw_generated": gen_pack["raw_generated"],
                "prediction": pred,
                "mean_logprob": gen_pack["mean_logprob"],
            }
            row.update(eval_pack)

            # last-layer summary
            for metric_name, scores in prompt_scores.items():
                row[f"prompt_last_{metric_name}"] = scores[-1]
            for metric_name, scores in output_scores.items():
                row[f"output_last_{metric_name}"] = scores[-1]

            all_outputs.append(row)

            n_layers = len(prompt_vecs)
            for layer_idx in range(n_layers):
                all_layers.append({
                    "model_name": model_cfg["name"],
                    "model_label": model_cfg["label"],
                    "instruction_tuned": model_cfg["instruction_tuned"],
                    "example_id": ex["id"],
                    "has_error": ex["has_error"],
                    "condition": cond["condition"],
                    "layer": layer_idx,
                    "prompt_correction_mode": prompt_scores["correction_mode"][layer_idx],
                    "prompt_conditional_mode": prompt_scores["conditional_mode"][layer_idx],
                    "prompt_high_confidence": prompt_scores["high_confidence"][layer_idx],
                    "prompt_low_confidence": prompt_scores["low_confidence"][layer_idx],
                    "output_correction_mode": output_scores["correction_mode"][layer_idx],
                    "output_conditional_mode": output_scores["conditional_mode"][layer_idx],
                    "output_high_confidence": output_scores["high_confidence"][layer_idx],
                    "output_low_confidence": output_scores["low_confidence"][layer_idx],
                })

df_outputs = pd.DataFrame(all_outputs)
df_layers = pd.DataFrame(all_layers)

print("df_outputs shape:", df_outputs.shape)
print("df_layers shape:", df_layers.shape)
display(df_outputs.head())


In [ ]:
# Cell 11: 保存
outputs_csv = DATA_DIR / "grammar_outputs.csv"
layers_csv = DATA_DIR / "grammar_layers.csv"

df_outputs.to_csv(outputs_csv, index=False)
df_layers.to_csv(layers_csv, index=False)

print("saved:", outputs_csv)
print("saved:", layers_csv)


## 出力評価


In [ ]:

# Cell 12: 出力評価サマリー
summary_rows = []

for (model_label, condition), sub in df_outputs.groupby(["model_label", "condition"]):
    err = sub[sub["has_error"] == True]
    cor = sub[sub["has_error"] == False]

    summary_rows.append({
        "model_label": model_label,
        "condition": condition,
        "n_total": len(sub),
        "n_error": len(err),
        "n_correct": len(cor),
        "exact_match_rate": sub["exact_match_gold"].mean(),
        "unchanged_from_input_rate": sub["unchanged_from_input"].mean(),
        "changed_rate": sub["changed"].mean(),
        "corrected_when_needed_rate": err["corrected_when_needed"].mean() if len(err) else np.nan,
        "preserved_when_correct_rate": cor["preserved_when_correct"].mean() if len(cor) else np.nan,
        "mean_semantic_to_gold": sub["semantic_to_gold"].mean(),
        "mean_semantic_to_input": sub["semantic_to_input"].mean(),
        "mean_lexical_change_ratio": sub["lexical_change_ratio"].mean(),
        "mean_grammar_acceptability": sub["grammar_acceptability"].mean(),
        "mean_logprob": sub["mean_logprob"].mean(),
    })

df_summary = pd.DataFrame(summary_rows).sort_values(["model_label", "condition"]).reset_index(drop=True)
display(df_summary)


summary_df = pd.DataFrame(summary_rows).sort_values(["model_label", "condition"]).reset_index(drop=True)
summary_csv = TABLE_DIR / "output_summary.csv"
summary_df.to_csv(summary_csv, index=False)

print("saved:", summary_csv)
display(summary_df)


In [ ]:

# Cell 13: A/B の paired comparison
pivot = df_outputs.pivot_table(
    index=["model_label", "example_id", "has_error"],
    columns="condition",
    values=[
        "exact_match_gold",
        "unchanged_from_input",
        "changed",
        "corrected_when_needed",
        "preserved_when_correct",
        "semantic_to_gold",
        "semantic_to_input",
        "lexical_change_ratio",
        "grammar_acceptability",
        "mean_logprob",
        "prompt_last_correction_mode",
        "prompt_last_conditional_mode",
        "prompt_last_high_confidence",
        "prompt_last_low_confidence",
        "output_last_correction_mode",
        "output_last_conditional_mode",
        "output_last_high_confidence",
        "output_last_low_confidence",
    ],
)
pivot.columns = ["__".join(col) for col in pivot.columns]
pivot = pivot.reset_index()

comparison_specs = [
    ("exact_match_gold__A_assert_error", "exact_match_gold__B_if_error", "exact_match_gold"),
    ("unchanged_from_input__A_assert_error", "unchanged_from_input__B_if_error", "unchanged_from_input"),
    ("changed__A_assert_error", "changed__B_if_error", "changed"),
    ("semantic_to_gold__A_assert_error", "semantic_to_gold__B_if_error", "semantic_to_gold"),
    ("semantic_to_input__A_assert_error", "semantic_to_input__B_if_error", "semantic_to_input"),
    ("lexical_change_ratio__A_assert_error", "lexical_change_ratio__B_if_error", "lexical_change_ratio"),
]

paired_rows = []
for model_label, sub in pivot.groupby("model_label"):
    for col_a, col_b, label in comparison_specs:
        row = paired_test_from_pivot(sub, col_a, col_b, label)
        row["model_label"] = model_label
        paired_rows.append(row)

df_paired = pd.DataFrame(paired_rows).sort_values(["model_label", "comparison"]).reset_index(drop=True)
display(df_paired)


paired_csv = TABLE_DIR / "paired_comparison.csv"
paired_df.to_csv(paired_csv, index=False)

print("saved:", paired_csv)
display(paired_df)


In [ ]:

# Cell 14: 代表出力の確認
display_cols = [
    "model_label", "example_id", "has_error", "condition",
    "sentence", "gold", "prediction",
    "exact_match_gold", "unchanged_from_input",
    "semantic_to_gold", "semantic_to_input",
]
display(df_outputs[display_cols].sort_values(["model_label", "example_id", "condition"]).head(30))



## レイヤー解析
以下を追加しています。

1. 正文 / 誤文を分けた層推移  
2. A-B 差分曲線  
3. 最大差 layer の要約表  


In [ ]:

# Cell 15: 正文 / 誤文を分けた prompt 側レイヤー推移
PROMPT_METRICS = [
    "prompt_correction_mode",
    "prompt_conditional_mode",
    "prompt_high_confidence",
    "prompt_low_confidence",
]

for model_label in df_layers["model_label"].unique():
    for has_error_value, split_name in [(True, "error_cases"), (False, "correct_cases")]:
        sub = df_layers[(df_layers["model_label"] == model_label) & (df_layers["has_error"] == has_error_value)].copy()
        plot_df = (
            sub.groupby(["condition", "layer"])[PROMPT_METRICS]
            .mean()
            .reset_index()
        )

        for metric in PROMPT_METRICS:
            plt.figure(figsize=(9, 4.8))
            for cond in sorted(plot_df["condition"].unique()):
                s = plot_df[plot_df["condition"] == cond]
                plt.plot(s["layer"], s[metric], label=cond)
            plt.axhline(0, linestyle="--")
            plt.xlabel("Layer")
            plt.ylabel("Cosine similarity")
            plt.title(f"{metric} | {model_label} | {split_name}")
            plt.legend()
            plt.show()


In [ ]:

# Cell 16: 正文 / 誤文を分けた output 側レイヤー推移
OUTPUT_METRICS = [
    "output_correction_mode",
    "output_conditional_mode",
    "output_high_confidence",
    "output_low_confidence",
]

for model_label in df_layers["model_label"].unique():
    for has_error_value, split_name in [(True, "error_cases"), (False, "correct_cases")]:
        sub = df_layers[(df_layers["model_label"] == model_label) & (df_layers["has_error"] == has_error_value)].copy()
        plot_df = (
            sub.groupby(["condition", "layer"])[OUTPUT_METRICS]
            .mean()
            .reset_index()
        )

        for metric in OUTPUT_METRICS:
            plt.figure(figsize=(9, 4.8))
            for cond in sorted(plot_df["condition"].unique()):
                s = plot_df[plot_df["condition"] == cond]
                plt.plot(s["layer"], s[metric], label=cond)
            plt.axhline(0, linestyle="--")
            plt.xlabel("Layer")
            plt.ylabel("Cosine similarity")
            plt.title(f"{metric} | {model_label} | {split_name}")
            plt.legend()
            plt.show()


In [ ]:

# Cell 17: A-B 差分曲線（prompt / output）
def build_delta_curve(df_layers_sub, metric, a_name="A_assert_error", b_name="B_if_error"):
    pivot = df_layers_sub.pivot_table(
        index=["example_id", "layer"],
        columns="condition",
        values=metric,
    ).reset_index()

    if a_name not in pivot.columns or b_name not in pivot.columns:
        return pd.DataFrame(columns=["layer", "delta_mean", "delta_se"])

    pivot["delta"] = pivot[a_name] - pivot[b_name]  # A - B
    agg = (
        pivot.groupby("layer")["delta"]
        .agg(["mean", "std", "count"])
        .reset_index()
        .rename(columns={"mean": "delta_mean", "std": "delta_std", "count": "n"})
    )
    agg["delta_se"] = agg["delta_std"] / np.sqrt(agg["n"].clip(lower=1))
    return agg

all_metrics = PROMPT_METRICS + OUTPUT_METRICS

for model_label in df_layers["model_label"].unique():
    for has_error_value, split_name in [(True, "error_cases"), (False, "correct_cases")]:
        sub = df_layers[(df_layers["model_label"] == model_label) & (df_layers["has_error"] == has_error_value)].copy()

        for metric in all_metrics:
            delta_df = build_delta_curve(sub, metric)
            if len(delta_df) == 0:
                continue

            plt.figure(figsize=(9, 4.8))
            plt.plot(delta_df["layer"], delta_df["delta_mean"])
            plt.fill_between(
                delta_df["layer"],
                delta_df["delta_mean"] - 1.96 * delta_df["delta_se"],
                delta_df["delta_mean"] + 1.96 * delta_df["delta_se"],
                alpha=0.2,
            )
            plt.axhline(0, linestyle="--")
            plt.xlabel("Layer")
            plt.ylabel("A - B")
            plt.title(f"Delta curve | {metric} | {model_label} | {split_name}")
            plt.show()


In [ ]:

# Cell 18: 最大差 layer の要約表
summary_rows = []

for model_label in df_layers["model_label"].unique():
    for has_error_value, split_name in [(True, "error_cases"), (False, "correct_cases")]:
        sub = df_layers[(df_layers["model_label"] == model_label) & (df_layers["has_error"] == has_error_value)].copy()

        for metric in all_metrics:
            delta_df = build_delta_curve(sub, metric)
            if len(delta_df) == 0:
                continue

            idx = delta_df["delta_mean"].abs().idxmax()
            best = delta_df.loc[idx]

            summary_rows.append({
                "model_label": model_label,
                "split": split_name,
                "metric": metric,
                "max_abs_delta_layer": int(best["layer"]),
                "delta_at_max_abs": float(best["delta_mean"]),
                "abs_delta": float(abs(best["delta_mean"])),
                "se_at_max_abs": float(best["delta_se"]),
            })

df_layer_delta_summary = pd.DataFrame(summary_rows).sort_values(
    ["model_label", "split", "abs_delta"],
    ascending=[True, True, False],
).reset_index(drop=True)

display(df_layer_delta_summary)
df_layer_delta_summary.to_csv(os.path.join(SAVE_DIR, "layer_delta_summary.csv"), index=False)


max_delta_csv = TABLE_DIR / "max_delta_layer_summary.csv"
max_delta_summary.to_csv(max_delta_csv, index=False)

print("saved:", max_delta_csv)
display(max_delta_summary)



## 読み方の目安

### 出力評価
- `exact_match_gold`: gold 完全一致
- `unchanged_from_input`: 原文そのまま返したか
- `semantic_to_gold`: gold との意味近さ
- `semantic_to_input`: 原文との意味近さ
- `grammar_acceptability`: 文法的に受理されそうか（optional）

### レイヤー解析
- `prompt_*`: プロンプトを読んだ時点の内部表現
- `output_*`: 生成結果テキスト側の内部表現
- `A - B > 0`: A の方がその metric に強い
- `A - B < 0`: B の方がその metric に強い

### 最初に見るべき点
1. instruct モデルで `preserved_when_correct_rate` が改善したか
2. `semantic_to_gold` と `semantic_to_input` がどう動いたか
3. 正文ケースで `prompt_conditional_mode` や `prompt_low_confidence` の差が出るか
4. 最大差が浅い層か深い層か


In [ ]:
# Cell 19: 出力ファイル一覧
artifact_rows = []
for base_dir in [DATA_DIR, TABLE_DIR, FIG_DIR, LOG_DIR]:
    for path in sorted(base_dir.rglob("*")):
        if path.is_file():
            artifact_rows.append({
                "category": path.parent.name,
                "relative_path": str(path.relative_to(RUN_DIR)),
                "size_kb": round(path.stat().st_size / 1024, 2),
            })

artifact_df = pd.DataFrame(artifact_rows)
artifact_csv = TABLE_DIR / "artifact_manifest.csv"
artifact_df.to_csv(artifact_csv, index=False)

print("saved:", artifact_csv)
display(artifact_df)

In [ ]:
# Cell 20: フォルダを zip にまとめる
zip_base = BASE_OUTPUT_DIR / RUN_NAME
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=BASE_OUTPUT_DIR, base_dir=RUN_NAME)
print("created zip:", zip_path)